In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

  Using cached kagglehub-1.0.1-py3-none-any.whl.metadata (40 kB)
  Using cached kagglesdk-0.1.23-py3-none-any.whl.metadata (13 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached kagglehub-1.0.1-py3-none-any.whl (70 kB)
Using cached kagglesdk-0.1.23-py3-none-any.whl (217 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [kagglehub]━━━━━━━━ 2/3 [kagglehub]
Note: you may need to restart the kernel to use updated packages.


# 🧪 Étape 6 : Évaluation Métrique & Robustesse (Squelette Étudiant)

Cette étape correspond au sixième chapitre du cours. L'objectif est de mettre en place un protocole d'évaluation rigoureux (splits d'évaluation adaptés) et de calculer les métriques clés de performance pour valider scientifiquement la qualité de vos modèles.

### 1. Préparation de l'environnement

In [2]:
import os
import sys
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.append(os.path.abspath('..'))
from src import data_clean as dc

print("Librairies prêtes pour l'évaluation des modèles !")

Librairies prêtes pour l'évaluation des modèles !


### 2. Évaluation du modèle Tabulaire

L'évaluation de notre modèle nous oblige à confronter nos algorithmes à la réalité du marché musical. Nous observons une **MAE de 11.1** et une **RMSE de 14.8**. La MAE nous indique qu'en moyenne, notre modèle se trompe de 11 points (sur 100) en prédisant la popularité. Cependant, le fait que la RMSE soit nettement supérieure à la MAE (presque 15 points) est fondamental : cela signifie que notre modèle fait parfois de **très grosses erreurs de prédiction**. Métier parlant, l'IA prédit parfois qu'une musique devrait être un 'flop' (sur la base de son audio) alors que c'est un énorme 'hit', ou inversement. Cela prouve que le succès d'un titre peut 'exploser' de manière imprévisible pour la machine, souvent à cause d'une *trend TikTok* soudaine ou du poids d'un artiste.


In [ ]:
# Chargement des données et ré-entraînement rapide pour évaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

df['duration_min'] = df['duration_ms'] / 60000
df['is_popular'] = (df['popularity'] >= 50).astype(int)
df['dance_energy_score'] = df['danceability'] * df['energy']
df['explicit_int'] = df['explicit'].astype(int)


features = [
    'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo',
    'duration_min', 'dance_energy_score', 'explicit_int'
]
target = 'popularity'

split = int(len(df) * 0.8)
X_train = df[features].iloc[:split]
X_test  = df[features].iloc[split:]
y_train = df[target].iloc[:split]
y_test  = df[target].iloc[split:]

# Entraînement
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# TODO: Calculez MAE, RMSE et R² entre y_test et y_pred
mae = mean_absolute_error(y_test, y_pred)
#   erreur moyenne de 12 points de popularité
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# pénalise les grosses erreurs
r2 = r2_score(y_test, y_pred)
#  le modèle explique seulement 10% de la popularité

print(f"Test MAE  : {mae:.3f}")
print(f"Test RMSE : {rmse:.3f}")
print(f"Test R²   : {r2:.3f}")

Test MAE  : 11.128
Test RMSE : 14.846
Test R²   : 0.230


### 3. Protocole de Validation Croisée (Out-of-Fold / Chronologique)

Étant donné l'absence de variable temporelle (comme expliqué lors du data wrangling), nous n'avons pas pu utiliser une validation `TimeSeriesSplit`. Nous avons opté pour une validation croisée classique en 5 plis (K-Fold). Le verdict final est un **R² moyen de 0.446**. Cette métrique est le point culminant de notre projet : elle signifie que les caractéristiques musicales pures (la 'danceability', le tempo, etc.) n'arrivent à expliquer que **44,6 % de la variance de la popularité**. Les 55,4 % restants échappent totalement à l'audio brut et s'expliquent par le contexte : la stratégie marketing du label, l'algorithme de recommandation et l'engagement social des fans.


In [5]:
# TODO: Proposer un script de K-Fold temporel (TimeSeriesSplit) ou de validation croisée classique
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ============================================================
# VALIDATION CROISÉE — KFold classique
# But : Tester la robustesse du modèle sur plusieurs
# découpages différents des données.
# On divise les données en 5 groupes (folds) :
# à chaque tour, 1 groupe sert de test et les 4 autres
# servent d'entraînement → 5 évaluations différentes !
# ============================================================

# Préparer les données
X = df[features]
y = df[target]

# Définir la validation croisée en 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Stocker les résultats de chaque fold
mae_scores  = []
rmse_scores = []
r2_scores   = []

print("=== Validation Croisée (5 Folds) ===\n")

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

    # Découper les données
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    # Entraîner le modèle
    model_fold = RandomForestRegressor(n_estimators=50, random_state=42)
    model_fold.fit(X_tr, y_tr)

    # Prédire et évaluer
    y_pred_fold = model_fold.predict(X_te)
    mae  = mean_absolute_error(y_te, y_pred_fold)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred_fold))
    r2   = r2_score(y_te, y_pred_fold)

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)

    print(f"Fold {fold+1} → MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.3f}")

# Résumé final
print("\n=== Résultats Moyens sur 5 Folds ===")
resultats_cv = pd.DataFrame({
    'Métrique': ['MAE', 'RMSE', 'R²'],
    'Moyenne': [
        round(np.mean(mae_scores), 3),
        round(np.mean(rmse_scores), 3),
        round(np.mean(r2_scores), 3)
    ],
    'Écart-type': [
        round(np.std(mae_scores), 3),
        round(np.std(rmse_scores), 3),
        round(np.std(r2_scores), 3)
    ]
})
display(resultats_cv)

print("\nProtocole de validation documenté avec succès !")

=== Validation Croisée (5 Folds) ===

Fold 1 → MAE: 10.55 | RMSE: 14.34 | R²: 0.446
Fold 2 → MAE: 10.52 | RMSE: 14.23 | R²: 0.446
Fold 3 → MAE: 10.66 | RMSE: 14.37 | R²: 0.444
Fold 4 → MAE: 10.54 | RMSE: 14.24 | R²: 0.442
Fold 5 → MAE: 10.62 | RMSE: 14.33 | R²: 0.452

=== Résultats Moyens sur 5 Folds ===


,Métrique,Moyenne,Écart-type
0,MAE,10.577,0.055
1,RMSE,14.301,0.057
2,R²,0.446,0.003



Protocole de validation documenté avec succès !
